In [1]:
import pandas as pd
import os

# Directory and files
dir_path = 'Decon-Results_Toden_NCI/'
files = [
    '20250405_GeneID_LessTissueV2_2Median-withGTFNames_All-NCI_Counts_v46_Clean_UniquePatients_CPM_BayesPrism.txt',
    '20250405_GeneID_LessTissueV2_2Median-withGTFNames_MuSiC_proportions.txt',
    'CIBERSORTx_Results.txt',
    'NNLS_All-NCI_Counts_v46_Clean_UniquePatients_CPM.txt',
    'nuSVR_CountsRMSE_All-NCI_Counts_v46_Clean_UniquePatients_CPM.txt',
    'QP_All-NCI_Counts_v46_Clean_UniquePatients_CPM_composition.txt',
    '20250405_GeneID_LessTissueV2_Sampling10-Unique_Top500_ReDeconv_results.tsv'
]

# Column renaming mapping
column_rename_map = {
    'Whole_blood': 'Whole blood',
    'Adrenal_gland': 'Adrenal gland'
}

# Target columns
target_columns = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast',
    'FemaleReproductive', 'Fibroblasts', 'Lymphocytes', 'ColonSigmoid',
    'ColonTransverse', 'Esophagus', 'EsophagusMucosa', 'Heart', 'Kidney',
    'Liver', 'Lung', 'SalivaryGland', 'MuscleSkeletal', 'NerveTibial',
    'Pancreas', 'Pituitary', 'Prostate', 'Skin', 'SmallIntestine', 'Spleen',
    'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Mapping of file name keywords to deconvolution tools
tool_map = {
    'BayesPrism': 'BayesPrism',
    'MuSiC': 'MuSiC',
    'CIBERSORTx': 'CIBERSORTx',
    'NNLS': 'NNLS',
    'nuSVR': 'nuSVR',
    'QP': 'QP',
    'ReDeconv': 'ReDeconv'
}

# Container for all dataframes
all_dfs = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = pd.read_csv(file_path, sep='\t', index_col=0)
    
    # Rename columns according to mapping
    df.rename(columns=column_rename_map, inplace=True)
    
    # Subset to keep only target columns that exist in the file
    available_columns = [col for col in target_columns if col in df.columns]
    df_subset = df[available_columns]
    
    # Normalise rows to sum to 100 and round to 2 decimal places
    df_normalised = df_subset.div(df_subset.sum(axis=1), axis=0) * 100
    df_normalised = df_normalised.round(2)
    
    # Efficient tool detection
    tool = next((v for k, v in tool_map.items() if k in file), None)
    if tool is None:
        raise ValueError(f"Could not determine deconvolution tool for file: {file}")
    
    # Add deconvolution tool as a new column
    df_normalised['DeconvolutionTool'] = tool

    all_dfs.append(df_normalised)

# Concatenate all dataframes
merged_df = pd.concat(all_dfs)

# Save merged file
output_file = os.path.join(dir_path, 'merged_normalised_results.txt')
merged_df.to_csv(output_file, sep='\t')

print(f'Merged and saved all results to: {output_file}')

Merged and saved all results to: Decon-Results_Toden_NCI/merged_normalised_results.txt
